In [5]:
!pip install icalendar python-dateutil

In [6]:
!wget https://raw.githubusercontent.com/mattsmi/Python_Calendar_Calcs/master/MattaCalendarCalculations.py

--2026-06-23 07:55:45--  https://raw.githubusercontent.com/mattsmi/Python_Calendar_Calcs/master/MattaCalendarCalculations.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8137 (7.9K) [text/plain]
Saving to: ‘MattaCalendarCalculations.py.1’

MattaCalendarCalcul 100%[===================>]   7.95K  --.-KB/s    in 0s      

2026-06-23 07:55:45 (74.8 MB/s) - ‘MattaCalendarCalculations.py.1’ saved [8137/8137]



In [7]:
import uuid
import datetime
from icalendar import Calendar, Event
from dateutil.easter import easter

# Import the repository file (Make sure MattaCalendarCalculations.py is in the same folder)
import MattaCalendarCalculations as mcc

def create_orthodox_feasts_ics(start_date, end_date, filename="orthodox_feasts.ics"):
    # Initialize the calendar
    cal = Calendar()
    cal.add('prodid', '-//Greek Orthodox Feast Days//EN')
    cal.add('version', '2.0')
    cal.add('calscale', 'GREGORIAN')

    # Fixed Feasts Dictionary (Revised Julian Month, Revised Julian Day: Name)
    fixed_feasts = {
        (1, 6): "Theophany",
        (2, 2): "Presentation of Christ",
        (3, 25): "Annunciation",
        (8, 6): "Transfiguration of Christ",
        (8, 15): "Dormition of the Theotokos",
        (9, 8): "Nativity of the Theotokos",
        (9, 14): "Elevation of the Holy Cross",
        (11, 21): "Presentation of the Theotokos",
        (12, 25): "Nativity of Christ"
    }

    # 1. Process Fixed Feasts by converting each Gregorian day to Revised Julian
    current_date = start_date
    while current_date <= end_date:
        # Convert Gregorian day to CJDN (Chronological Julian Day Number)
        cjdn = mcc.pGregorianToCJDN(current_date.year, current_date.month, current_date.day)

        # Convert CJDN to Revised Julian (Milanković) to extract month and day
        rj_month = mcc.pCJDNToMilankovic(cjdn, False, True, False)
        rj_day = mcc.pCJDNToMilankovic(cjdn, False, False, True)

        # Check if the Revised Julian month and day match any of our fixed feasts
        if (rj_month, rj_day) in fixed_feasts:
            feast_name = fixed_feasts[(rj_month, rj_day)]
            _add_event(cal, feast_name, current_date)

        current_date += datetime.timedelta(days=1)

    # 2. Process Movable Feasts (Pascha-based)
    # We iterate through the unique years present in the date range
    for year in range(start_date.year, end_date.year + 1):
        # dateutil.easter method=2 uses the Orthodox (Julian) calculation but returns a Gregorian date
        pascha_date = easter(year, method=2)

        movable_feasts = [
            (pascha_date - datetime.timedelta(days=48), "Clean Monday"),
            (pascha_date - datetime.timedelta(days=7), "Palm Sunday"),
            (pascha_date, "Pascha"),
            (pascha_date + datetime.timedelta(days=39), "Ascension of Christ"),
            (pascha_date + datetime.timedelta(days=49), "Holy Pentecost")
        ]

        for event_date, name in movable_feasts:
            # Only add the movable feast if it falls within the requested date range
            if start_date <= event_date <= end_date:
                _add_event(cal, name, event_date)

    # Write the calendar to a file
    with open(filename, 'wb') as f:
        f.write(cal.to_ical())

    print(f"Successfully generated '{filename}' from {start_date} to {end_date}.")

def _add_event(cal, summary, event_date):
    """Helper function to create an all-day event and add it to the calendar."""
    event = Event()
    event.add('summary', summary)

    # Adding a datetime.date object automatically creates an all-day event
    event.add('dtstart', event_date)
    event.add('dtend', event_date + datetime.timedelta(days=1))

    event.add('dtstamp', datetime.datetime.now())
    event.add('uid', str(uuid.uuid4()) + '@orthodoxfeasts.local')

    cal.add_component(event)

if __name__ == "__main__":
    # Define your Gregorian date range here
    START_DATE = datetime.date(2026, 1, 1)
    END_DATE = datetime.date(2030, 12, 31)
    OUTPUT_FILE = "orthodox-feasts-2026-2030.ics"

    create_orthodox_feasts_ics(START_DATE, END_DATE, OUTPUT_FILE)

Successfully generated 'orthodox-feasts-2026-2030.ics' from 2026-01-01 to 2030-12-31.
